This is the reproduction notebook for the main PCSS + MiSS experiment for `Qwen3.5-9B-Base`.

This specific notebook was tested on this docker image on a H100 SXM: `vastai/pytorch:2.10.0-cuda-13.0.2-py312-24.04-2026-03-26`

It takes about ~40 minutes to train on a H100 SXM / H200 NVL. You can use other hardware for sure, just make sure to switch `attn_implementation` accordingly (e.g., to 'sdpa')

If you are using the same docker image and hopper hardware (with cu130 compatibility), you can just run all cells. When you're connecting to the jupyter server, select `main venv`

We use a custom transformers fork for dense Qwen 3.5 MTP training support: https://github.com/tamewild/transformers/tree/verify-qwen-mtp-implementation

##### Installation

Here we install `uv` and prerequisites:

In [ ]:
%pip install uv
!/venv/main/bin/python -m uv pip install "git+https://github.com/tamewild/transformers.git@7ccc0ddb811422a86c7737a5497e05352a1c9ae3" trl==1.4.0 peft==0.19.1 bitsandbytes==0.49.2 kernels==0.14.1
!/venv/main/bin/python -m uv pip install git+https://github.com/linkedin/Liger-Kernel.git@30b8486a2bd48dff97f22c5f0c88520b8825cb35
!/venv/main/bin/python -m uv pip install flash-linear-attention==0.5.0
!/venv/main/bin/python -m uv pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu13torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!/venv/main/bin/python -m uv pip install tilelang==0.1.11 apache-tvm-ffi==0.1.11

In case that something goes wrong, make sure `%pip freeze` matches the following output:

In [ ]:
%pip freeze

In [ ]:
import os, torch, copy
from transformers import AutoModelForMultimodalLM, AutoTokenizer
from liger_kernel.transformers import apply_liger_kernel_to_qwen3_5
from typing import Optional, List
import torch
import torch.nn as nn

apply_liger_kernel_to_qwen3_5() # apply here so we benefit during inference

model = AutoModelForMultimodalLM.from_pretrained(
    "Qwen/Qwen3.5-9B-Base",
    attn_implementation = "kernels-community/flash-attn3@v1",
    dtype = torch.bfloat16,
    device_map = "cuda"
)
model.config.text_config.use_cache = False
# use unsloth fixed official chat template
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-9B")

##### Dataset Processing

In [ ]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from typing import Union, List, Tuple

def create_pcss_mtp_dataset(
    dataset: Union[Dataset, DatasetDict],
    tokenizer: AutoTokenizer,
    ref_model: nn.Module
) -> Union[Dataset, DatasetDict]:
    ref_model.eval()
    
    def _process_single_example(example: dict) -> dict:
        prompt_str = tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": example["prompt"]
                }
            ],
            add_generation_prompt=True,
            enable_thinking=False,
            tokenize=False
        )
        full_str = prompt_str + example["completion"] + tokenizer.eos_token

        prompt_tokens = tokenizer(prompt_str, return_tensors="pt")
        full_tokens = tokenizer(full_str, return_tensors="pt")
        prompt_len = prompt_tokens.input_ids.shape[1]

        labels = full_tokens.input_ids.clone()
        labels[:, :prompt_len] = -100

        inputs_for_ref = {
            "input_ids": full_tokens.input_ids.to(ref_model.device),
            "attention_mask": full_tokens.attention_mask.to(ref_model.device),
            "labels": labels.to(ref_model.device),
            "num_mtp_steps": 1
        }

        with torch.no_grad():
            outputs = ref_model.forward_mtp(**inputs_for_ref)
            losses = [outputs.loss_0] + outputs.mtp_losses
            l_refs = [l.cpu().item() for l in losses]

        return {
            "input_ids": full_tokens.input_ids.squeeze(0).tolist(),
            "attention_mask": full_tokens.attention_mask.squeeze(0).tolist(),
            "labels": labels.squeeze(0).tolist(),
            "l_refs": l_refs,
        }

    processed_dataset = None
    if isinstance(dataset, DatasetDict):
        processed_splits = {}
        for split_name, split_dataset in dataset.items():
            processed_examples = [
                _process_single_example(ex)
                for ex in tqdm(split_dataset, desc=f"Processing split '{split_name}'")
            ]
            processed_splits[split_name] = Dataset.from_list(processed_examples)
        processed_dataset = DatasetDict(processed_splits)
    elif isinstance(dataset, Dataset):
        processed_examples = [
            _process_single_example(ex)
            for ex in tqdm(dataset, desc="Processing dataset")
        ]
        processed_dataset = Dataset.from_list(processed_examples)
    else:
        raise TypeError(f"Input must be a Dataset or DatasetDict, but got {type(dataset)}")

    return processed_dataset

dataset = load_dataset("tamewild/instruct5")

dataset = dataset.map(lambda example: {
    "prompt": example["conversation"][0]["content"],
    "completion": example["conversation"][1]["content"]
})

dataset = create_pcss_mtp_dataset(
    dataset,
    tokenizer,
    model
)

##### Dataset Inspection

I recommend using HF's dataset inspector on the website, it's convenient. But here you can inspect the first entry.

See first entry without prompt masked:

In [ ]:
print(tokenizer.decode(dataset['train'][0]['input_ids']))

See first entry with prompt masked:

In [ ]:
def print_unmasked_part(entry):
    unmasked = list(filter(lambda label: label != -100, entry["labels"]))
    print(tokenizer.decode(unmasked))

print_unmasked_part(dataset['train'][0])

##### Training

In [ ]:
from peft import get_peft_model, MissConfig

model = get_peft_model(
    model,
    MissConfig(
        r = 512,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
            "in_proj_a", "in_proj_b", "in_proj_qkv", "in_proj_z", "out_proj",
            "fc"
        ],
        bias = "none",
        modules_to_save = None,
        task_type="CAUSAL_LM",
        init_weights = True # "MiSS efficience and balance"
    ),
    autocast_adapter_dtype = False # stay in bf16
)

model.print_trainable_parameters()

In [ ]:
from transformers import Trainer
import torch
import torch.nn as nn
from typing import Dict, Union, Any, Optional

class PCSSTrainer(Trainer):
    def __init__(self, *args, beta, peak_scale, global_l_std, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        self.peak_scale = peak_scale
        self.global_l_std = global_l_std

    def compute_loss(
        self,
        model: nn.Module,
        inputs: Dict[str, Union[torch.Tensor, Any]],
        return_outputs: bool = False,
        num_items_in_batch: Optional[torch.Tensor] = None,
    ):
        l_refs = inputs.pop("l_refs").view(-1)

        mtp_outputs = model.forward_mtp(**inputs, num_mtp_steps=1)

        mode = "train" if self.model.training else "eval"

        if mode == "train":
            # Base Model (Step 0)
            l_sft_0 = mtp_outputs.loss_0
            l_ref_0 = l_refs[0]
            std_0 = self.global_l_std[0]

            with torch.no_grad():
                advantage_0 = (l_ref_0 - l_sft_0) / std_0
                sigmoid_input_0 = self.beta * advantage_0
                sigmoid_val_0 = torch.sigmoid(sigmoid_input_0)
                sigmoid_prime_0 = sigmoid_val_0 * (1.0 - sigmoid_val_0)
                scale_0 = self.peak_scale * (4.0 * sigmoid_prime_0)

            scaled_loss_0 = l_sft_0 * scale_0.detach()

            # MTP Step 1
            l_sft_1 = mtp_outputs.mtp_losses[0]
            l_ref_1 = l_refs[1]
            std_1 = self.global_l_std[1]

            with torch.no_grad():
                advantage_1 = (l_ref_1 - l_sft_1) / std_1
                sigmoid_input_1 = self.beta * advantage_1
                sigmoid_val_1 = torch.sigmoid(sigmoid_input_1)
                sigmoid_prime_1 = sigmoid_val_1 * (1.0 - sigmoid_val_1)
                scale_1 = self.peak_scale * (4.0 * sigmoid_prime_1)

            scaled_loss_1 = l_sft_1 * scale_1.detach()

            final_loss = 0.5 * (scaled_loss_0 + scaled_loss_1)

            return (final_loss, mtp_outputs) if return_outputs else final_loss
        else:
            return (mtp_outputs.loss_0, mtp_outputs) if return_outputs else mtp_outputs.loss_0

In [ ]:
from transformers import TrainingArguments, default_data_collator

def std_calc(l_refs):
    tensor_refs = torch.tensor(l_refs)
    return torch.clamp(tensor_refs.std(dim=0), min=1e-6).tolist()

global_l_std = std_calc(dataset['train']['l_refs'])

print(f'Global Standard Deviations: {global_l_std}')

trainer = PCSSTrainer(
    model = model,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    data_collator = default_data_collator,
    beta=0.65,
    peak_scale=5,
    global_l_std=global_l_std,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 500, # fixed at 1 epoch
        num_train_epochs = 5,
        learning_rate = 5e-6,
        fp16 = False,
        bf16 = True,
        logging_strategy = "no",
        optim = "adamw_8bit",
        adam_beta2 = 0.99994, # scale optimizer memory. according to paper
        weight_decay = 0.01,
        lr_scheduler_type = "constant_with_warmup",
        seed = 3407,
        output_dir = "workspace",
        save_strategy = "steps",
        save_steps = 2500,
        eval_strategy = "epoch",
        gradient_checkpointing = False, # highly recommended for h100, h200, etc
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        use_liger_kernel = False,
        prediction_loss_only = True,
        remove_unused_columns = False
    )
)

In [ ]:
trainer.train()

In [ ]:
import gc
from peft import PeftModel

# free vram
del model, trainer
torch.cuda.empty_cache()
gc.collect()

# merge with `autocast_adapter_dtype=True` (fp32)
model = AutoModelForMultimodalLM.from_pretrained(
    "Qwen/Qwen3.5-9B-Base",
    attn_implementation = "sdpa",
    dtype = torch.bfloat16,
    device_map = "cuda"
)
model = PeftModel.from_pretrained(
    model,
    "/workspace/checkpoint-2500",
    autocast_adapter_dtype = True
)
model = model.merge_and_unload()

# save locally, you can optionally upload to HF (look it up) but please avoid polluting HF with essentially the same model trained in a few minutes
model.save_pretrained("./merged")
tokenizer.save_pretrained("./merged")

# free vram again
del model
gc.collect()
torch.cuda.empty_cache()

##### AIME 2025 Evaluation using vLLM

In [ ]:
# NOTE: This is for cu130, make sure your device supports it. Also pin fastapi because of https://github.com/vllm-project/vllm/issues/45597
!/venv/main/bin/python -m uv pip install vllm==0.19.1 fastapi==0.136.3 transformers==5.8.1 --extra-index-url https://wheels.vllm.ai/0.19.1/cu130
!/venv/main/bin/python -m uv pip install math-verify[antlr4_13_2]==0.9.0 #pinned version of hf math-verify

**NOTE**: You may have to restart kernel before running the following cell

In [ ]:
from datasets import load_dataset
from vllm import LLM, SamplingParams
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig

llm = LLM(
    model = "./merged",
    max_model_len = 17_000,
    gpu_memory_utilization = 0.85,
    language_model_only = True,
    generation_config = "vllm",
    disable_log_stats = False,
    speculative_config = {
        "method": "mtp",
        "num_speculative_tokens": 2,
    }
)

def extract_last_box(s: str) -> str:
    tag = "\\boxed{"
    tag_start = s.rfind(tag)
    if tag_start == -1:
        return ""

    content = s[tag_start + len(tag):]
    depth = 1
    for i, char in enumerate(content):
        if char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                return content[:i].strip()

    return ""

def check_math500(ground_truth, response) -> bool:
    boxed_content = extract_last_box(response)
    if not boxed_content:
        return False

    boxed_response = f"\\boxed{{{boxed_content}}}"

    # Ensure ground truth is parsable by math_verify
    if "\\boxed" not in ground_truth:
        ground_truth = f"\\boxed{{{ground_truth}}}"

    gold = parse(
        ground_truth,
        # https://github.com/huggingface/Math-Verify/blob/ba3d3aaff23b3f4cac7a14672b4f6e293d97c98b/src/math_verify/tasks.py#L219
        [LatexExtractionConfig(boxed_match_priority=0)]
    )
    resp = parse(
        boxed_response,
        # https://github.com/huggingface/Math-Verify/blob/ba3d3aaff23b3f4cac7a14672b4f6e293d97c98b/src/math_verify/tasks.py#L221
        [
            LatexExtractionConfig(boxed_match_priority=0),
            ExprExtractionConfig()
        ]
    )
    return verify(gold, resp)

def benchmark_aime2025(llm):
    aime2025_benchmark = load_dataset("MathArena/aime_2025")['train']

    extra = r"Please reason step by step, and put your final answer within \boxed{}"

    aime2025_benchmark = aime2025_benchmark.map(lambda example: {
        "problem": example["problem"].strip() + f"\n\n{extra}"
    })

    prompts = [[{"role": "user", "content": question["problem"]}] for question in aime2025_benchmark]

    prompt_outputs = llm.chat(
        prompts,
        SamplingParams(
            n=6,
            temperature=1.0,
            top_p=0.95,
            top_k=20,
            max_tokens=16_384
        ),
        chat_template_kwargs={"enable_thinking": False},
        use_tqdm=True
    )

    correct = 0

    for solution, prompt_output in zip(aime2025_benchmark['answer'], prompt_outputs):
        solution = str(solution)
        for output in prompt_output.outputs:
            if check_math500(solution, output.text):
                correct += 1

    print(f"Correct answers: {correct}")
    print(f"Estimated pass@1: {correct / (len(aime2025_benchmark['problem']) * 6)}")

benchmark_aime2025(llm)

Training is **not** deterministic and vLLM itself has non-determinism even with greedy decoding.

Due to longer training times, we only ran it twice (one for the ablation matrix and one for reproduction).

We got 60.67% and ~58% respectively

You can expect your reproduced score to land somewhere in this general neighborhood, which is well above the 49.44% base model baseline.